# Bài 2
Đây là notebook chứa mã nguồn đầy đủ của bài 2.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai02(budget=100, min_I=25, min_AI=15, coef_I=0.85, coef_AI=1.20, coef_H=0.95, coef_RD=1.35):
    labels = ['I Hạ tầng', 'AI Dữ liệu', 'H Nhân lực', 'R&D']

    # 1. Linprog solve
    x, z, ok = _solve(budget, min_I, min_AI, min_H=20, coef_I=coef_I, coef_AI=coef_AI, coef_H=coef_H, coef_RD=coef_RD)

    # 2. PuLP solve for dual values
    m = pulp.LpProblem("Budget_Allocation", pulp.LpMaximize)
    x1 = pulp.LpVariable('x1', lowBound=min_I)
    x2 = pulp.LpVariable('x2', lowBound=min_AI)
    x3 = pulp.LpVariable('x3', lowBound=20)
    x4 = pulp.LpVariable('x4', lowBound=10)

    m += coef_I*x1 + coef_AI*x2 + coef_H*x3 + coef_RD*x4, "Objective"
    m += x1 + x2 + x3 + x4 <= budget, "C1_Budget"
    m += -0.35*x1 + 0.65*x2 - 0.35*x3 + 0.65*x4 >= 0, "C2_Tech35"

    m.solve(pulp.PULP_CBC_CMD(msg=False))
    dual_values = {}
    if m.status == pulp.LpStatusOptimal:
        for name, c in m.constraints.items():
            dual_values[name] = round(c.pi, 4)

    # 3. Sensitivity: vary budget
    budgets = sorted(set([80, 100, 120, 140, budget]))
    sens_budgets = []
    sens_z = []
    for bb in budgets:
        sens_budgets.append(bb)
        sens_z.append(_solve(bb, min_I, min_AI, 20)[1])

    # 4. Scenario: x3 >= 30
    x_sc, z_sc, ok_sc = _solve(budget, min_I, min_AI, min_H=30, coef_I=coef_I, coef_AI=coef_AI, coef_H=coef_H, coef_RD=coef_RD)

    return {
        'status':      'Optimal' if ok else 'Infeasible',
        'Z':           z,
        'allocation':  dict(zip(labels, x)),
        'dual_values': dual_values,
        'sensitivity_budgets': sens_budgets,
        'sensitivity_z': sens_z,
        'scenario_x3': {
            'status': 'Optimal' if ok_sc else 'Infeasible',
            'Z': z_sc,
            'allocation': dict(zip(labels, x_sc)) if ok_sc else {}
        },
        'feasible':    ok,
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai02()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())